# **Analysis of Illegal Advertisements**
##### **Python version: 3.13.3**
---

### **Packages preparation**

##### Installation

In [ ]:
!pip install -r requirements.txt

##### Import

In [ ]:
import numpy as np
import pandas as pd
import os, faiss, json, pickle, time, openai, tiktoken
import google.generativeai as genai
from tqdm import tqdm
from dotenv import load_dotenv

### **Load data**

In [ ]:
load_dotenv()

In [ ]:
CASE_PATH = "formatted_cases.txt"
# CASE_PATH = "cases.json"
QUERY_PATH = "final_project_query.csv"

with open(CASE_PATH, "r") as f:
    case_list = f.read().split("\n\n")
    # case_list = json.load(f)

with open(QUERY_PATH, "r", encoding="utf-8") as f:
    query_list = list(pd.read_csv(f).iloc[:, 1])

### **Embedding**
The embedding part could be skipped if the embedding vectors have already been stored

##### Model and API_KEY setup

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-large"
openai.api_key = os.getenv("OPENAI_API_KEY")
MAX_TOKENS = 8192

##### Functions

In [ ]:
client = openai.OpenAI()
tokenizer = tiktoken.encoding_for_model(EMBEDDING_MODEL)

def truncate_text(text, max_length=MAX_TOKENS):
    tokens = tokenizer.encode(text)
    if len(tokens) > max_length:
        tokens = tokens[:max_length]
    return tokenizer.decode(tokens)

def embed_with_retry(text_list, retries=3, wait=10):
    for i in range(len(text_list)):
        text_list[i] = truncate_text(text_list[i]) # Truncate text if it exceeds max tokens

    for attempt in range(retries):
        try:
            response = client.embeddings.create(model=EMBEDDING_MODEL, input=text_list)
            return [np.array(e.embedding, dtype=np.float32) for e in response.data]
        except Exception as e:
            print(f"⚠️ error: {e}, retry {attempt + 1}/{retries}...")
            time.sleep(wait)
    raise RuntimeError("Failed to embed after multiple attempts.")

##### Case embedding

In [ ]:
case_embeddings = embed_with_retry(case_list)
# Store to FAISS
embedding_dim = len(case_embeddings[0])
index = faiss.IndexFlatL2(embedding_dim)
index.add(np.array(case_embeddings, dtype=np.float32))

##### Query embedding

In [ ]:
query_embeddings = embed_with_retry(query_list)

##### Save vectors

In [ ]:
INDEX_PATH = "embeddings/faiss_index.index"
QUERY_EMBEDDINGS_PATH = "embeddings/query.pkl"

os.makedirs("embeddings", exist_ok=True)
faiss.write_index(index, INDEX_PATH)
with open(QUERY_EMBEDDINGS_PATH, "wb") as f:
    pickle.dump(query_embeddings, f)

### **Retrieval**

##### Reload embeddings

In [ ]:
INDEX_PATH = "embeddings/faiss_index.index"
QUERY_EMBEDDINGS_PATH = "embeddings/query.pkl"

index = faiss.read_index(INDEX_PATH)
with open(QUERY_EMBEDDINGS_PATH, "rb") as f:
    query_embeddings = pickle.load(f)

##### Functions

In [ ]:
def retrieve_similar_cases(query_embedding, k=3):
    distances, indices = index.search(query_embedding.reshape(1, -1), k)
    return [case_list[i] for i in indices[0]]

In [ ]:
# Test
query = "本產品可以幫助調整腸胃道機能"
similar_cases = retrieve_similar_cases(embed_with_retry([query])[0])
for case in similar_cases:
    print(case, end="\n\n")

In [ ]:
# Test
similar_cases = retrieve_similar_cases(query_embeddings[2])
for case in similar_cases:
    print(case, end="\n\n")

### **Querying**

##### Model and API_KEY setup

In [ ]:
# LLM_MODEL = "gemini-2.5-flash-preview-05-20"
# genai.configure(api_key=os.getenv("GENAI_API_KEY"))
# llm_model = genai.GenerativeModel(LLM_MODEL)

LLM_MODEL = "gpt-4.1"
openai.api_key = os.getenv("OPENAI_API_KEY")
client = openai.OpenAI()

##### Prompt

In [ ]:
SYSTEM_PROMPT = """
你是一位廣告審查專家，請根據以下台灣法規來合規性判斷一則廣告內容的違法機率（若您認為違法機率 > 80% 則表示不合法），並解釋相關法條內容及原因。
請注意，由於廣告內容是爬蟲爬下來的，內容中可能會有許多較無關於其產品的資訊，比如無關的網站資訊等，
請不要分析並忽略這些無關產品資訊，專注於檢查廣告的核心內容。（但介紹產品內容可不限於使用中文，可能是使用其他國語言）
            
### 合規性判斷
- **無罪判定原則**：未找到確切違反法律的證據時，應推定其為無罪
- **從寬認定合法性**：不捏造或過度解讀廣告文字，例如推論產品「暗示」了哪些效用，應減少此類較無證據之推論，但如果是**藥物**宣稱**科學實證**、**國外研究**一類用語，則提高違法可能性認定，除非內容中出現完整的**衛福部核可字號**或**衛福部認證**。
- **比對允許使用的廣告用詞**：
    - 「調整體質」、「促進新陳代謝」、「保護消化道全機能」、「改變細菌叢生態」、「在嚴謹的營養均衡與熱量控制，以及適當的運動條件下，適量攝取本產品有助於不易形成體脂肪」
    - 「能完整補充人體營養」、「提升、調節生理機能」、「幫助入睡」、「青春美麗」、「排便超有感」、「給你排便順暢新體驗」、「改變細菌叢生態」、「提升吸收滋養消化機能」"這些文字出現時不視為有違法風險。
                            
## 分析步驟
1. **解析廣告內容**：先觀察分類廣告商品屬於哪類產品，接著檢視其是否涉及療效誇大。
2. **案例文件與法規比對**：根據以下法律文件與案例，提供比對結果。
3. **判斷違法機率**：依據法律及案例進行風險評估。
                            
## 法律
- 健康食品管理法
    - 第2條-第1項: 本法所稱健康食品，指具有保健功效，並標示或廣告其具該功效之食品。
    - 第2條-第2項: 本法所稱之保健功效，係指增進民眾健康、減少疾病危害風險，且具有實質科學證據之功效，非屬治療、矯正人類疾病之醫療效能，並經中央主管機關公告者。
    - 第4條
        - 健康食品之保健功效，應以下列方式之一表達︰
        - 一、如攝取某項健康食品後，可補充人體缺乏之營養素時，宣稱該食品具有預防或改善與該營養素相關疾病之功效。
        - 二、敘述攝取某種健康食品後，其中特定營養素、特定成分或該食品對人體生理結構或生理機能之影響。
        - 三、提出科學證據，以支持該健康食品維持或影響人體生理結構或生理機能之說法。
        - 四、敘述攝取某種健康食品後的一般性好處。
    - 第14條: 健康食品之標示或廣告，不得涉及醫療效能之內容。
                            
- 食品安全衛生管理法
    - 第3條: 食品：指供人食用或飲用之產品及其原料。（定義條款，無罰則）
    - 第28條-第1項: 食品、食品添加物、食品用洗滌劑及經中央主管機關公告之食品器具、食品容器或包裝，其標示、宣傳或廣告不得涉及醫療效能。                   

- 化粧品衛生管理法
    - 第10條-第1項: 化粧品之標示、宣傳及廣告內容，不得有虛偽或誇大之情事。
    - 第10條-第2項: 化粧品不得為醫療效能之標示、宣傳或廣告。
                            
- 藥事法
    - 第4條: 本法所稱藥物，係指藥品及醫療器材。
    - 第66條-第1項: 藥商刊播藥物廣告時，應於刊播前將所有文字、圖畫或言詞，申請中央或直轄市衛生主管機關核准，未經核准或經核准後廢止者，不得刊播。
    - 第66條-第2項: 藥物廣告在核准登記、刊播期間不得變更原核准事項。
    - 第66條-第3項: 傳播業者不得刊播未經中央或直轄市衛生主管機關核准、與核准事項不符、已廢止或經令立即停止刊播並限期改善而尚未改善之藥物廣告。
    - 第66條-第4項: 受委託刊播之傳播業者，應自廣告之日起六個月，保存委託刊播廣告者之姓名（公司或團體名稱）、身分證或事業登記證字號、住居所（事務所或營業所）及電話等資料，且於主管機關要求提供時，不得規避、妨礙或拒絕。
    - 第68條: 藥物廣告不得以左列方式為之：一、假借他人名義為宣傳者。二、利用書刊資料保證其效能或性能。三、藉採訪或報導為宣傳。四、以其他不正當方式為宣傳。
    - 第69條: 非本法所稱之藥物，不得為醫療效能之標示或宣傳。
                            
- 醫療器材管理法
    - 第6條: 醫療器材廣告指利用傳播方法，宣傳醫療效能，以達招徠銷售醫療器材為目的之行為。採訪、報導或宣傳之內容暗示或影射醫療器材之醫療效能，以達招徠銷售醫療器材為目的者，視為醫療器材廣告。
    - 第40條: 非醫療器材商不得為醫療器材廣告。
    - 第46條: 非醫療器材，不得為醫療效能之標示或宣傳。但其他法律另有規定者，不在此限。
"""

USER_PROMPT = """
以下是廣告內容，請分析廣告內容的違法機率？

## 廣告句
{ad}

## 與廣告內容相關之參考案例
{similar_cases}

請根據上述法律，「合規性判斷」廣告內容的違法機率，並提供相關法條內容及解釋。（第一行請只輸出"違法機率：<違法機率>"，不要有多餘文字）
    
## 輸出格式：
違法機率：<違法機率>

相關法條內容及解釋:
<說明>                            

## 回應結果範例

"範例1":
"prompt":
"請分析廣告內容：'本產品每日至少3~4次可隨餐食用，幫助維持消化道機能' 是否合法？"

"response":
"違法機率：10%
違法內容分析: 經分析廣告內容，違法風險較低。參考依據: 食品及相關產品標示宣傳廣告涉及不實誇張易生誤解或醫療效能認定準則-項次3"

"範例1":
"prompt":
"請分析廣告內容：'2023營養專家推薦益生菌酵素配方 最佳替代方案胃食道逆流' 是否合法？"

"response":
"違法機率：90%
違法內容分析: 段落或句子: "2023營養專家推薦益生菌酵素配方 最佳替代方案胃食道逆流", 由於承諾療效過度具體且容易誤導消費者對於醫療效益的期待。 違反條款: 違反《食品安全衛生管理法》第28條第1項。"
"""

##### Functions

In [ ]:
def extract_answer(response):
    line = response.splitlines()[0]
    if line.startswith("違法機率："):
        percent = int(line.split("：")[1].strip().replace("%", ""))
        answer = 0 if percent > 80 else 1
    else:
        answer = 0
    return answer

def query_with_retry(ad, ad_embedding, max_retries=3, retry_delay=5):
    system_prompt = SYSTEM_PROMPT
    # user_prompt = USER_PROMPT.format(ad=ad) # No RAG
    user_prompt = USER_PROMPT.format(ad=ad, similar_cases="\n\n".join(retrieve_similar_cases(ad_embedding)))
    # full_prompt = system_prompt + "\n" + user_prompt
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    retries = 0
    while retries < max_retries:
        try:
        # response = llm_model.generate_content(full_prompt).text.strip()
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=messages,
            ).choices[0].message.content.strip()
            return {"prompt": system_prompt + "\n" + user_prompt, "response": response, "answer": extract_answer(response)}
        except Exception as e:           
            retries += 1
            print(f"Some exception occurs, retrying {retries}/{max_retries} in {retry_delay} seconds...")
            time.sleep(retry_delay)
    raise RuntimeError(f"Failed to process row after {max_retries} retries.")

In [ ]:
# Test
ad = "酵素 - Google Shopping 三多消化酵素Plus膜衣錠60錠Visit siteNT$446.00 · 三多sentosa+NT$110.00 shippingCOMPARE PRICESPRODUCT DETAILS添加更多種不同功能性的酵素、含澱粉分解酵素、脂肪分解酵素、蛋白質分解酵素、鳳梨酵素、木瓜酵素,幫助食物中三大營養素(醣類、脂肪、蛋白質)分解與消化,有助於營養素吸收。 B群加強能量代謝:含維生素B1、B2、B6、泛酸,促進新陳代謝。 特別添加芽孢乳酸菌:可順利通過消化道,可幫助維持消化道機能、改變細菌叢生態、健康維持。 添加鎂,有助於身體正常代謝。 幫助消化 促進新陳代謝 品 名:三多消化酵素Plus膜衣錠 原料成分:微結晶狀α-纖維素、綜合酵素(麥芽糊精、澱粉分解酵素、植酸酵素、葡萄糖澱粉酵素、複合酵素(脂肪分解酵素、蛋白質分解酵素、澱粉分解酵素、麥芽糊精)、中性蛋白質分解酵素、鳳梨酵素、蛋白質分解酵素4.5、轉化酵素、蛋白質分解酵素3.0、半纖維素分解酵素、果膠酵素、α-半乳糖苷酶、蛋白質分解酵素、脂肪分解酵素、胜肽分解酵素、乳糖分解酵素、蛋白質分解酵素6.0、澱粉酵素、纖維素分解酵素)、綜合酵素Plus(麥芽糊精、澱粉分解酵素、蛋白質分解酵素、脂肪分解酵素、纖維分解酵素)、啤酒酵母(啤酒酵母、麥芽萃取)氧化鎂、青木瓜萃取物(麥芽糊精、乳糖、青木瓜萃取物、益生菌發酵物粉末(唾液乳桿菌發酵粉、乳雙岐桿菌發酵粉、長雙岐桿菌發酵粉、麥芽糊精、檸檬酸)羥丙基甲基纖維素、本多酸鈣、二氧化鈦、食用黃色四號鋁麗基、乳酸菌(Bacillus Coagulans、乳糖)維生素B1、維生素B2、維生素B6、食用黃色五號鋁麗基、食用藍色一號鋁麗基。 (食品添加物符合國家安全標準,請安心食用) 容量:0.5公克/錠,60錠/盒 保存期限:三年 有效日期:日/月/西元年 貨源: 公司貨 用法用量: 1、每餐飯後1錠。 2、請搭配溫冷開水食用。"
result = query_with_retry(ad, embed_with_retry([ad])[0])
print(result["answer"])
print(result["prompt"])
print(result["response"])

##### Query the LLM

In [ ]:
results = []
for id, query in tqdm(list(enumerate(query_list)), desc="Querying"):
    results.append(query_with_retry(query, query_embeddings[id]))
    print(id, results[id]["answer"])

##### Save results

In [ ]:
OUTPUT_PATH = "results/final_project_results.csv"
LOG_PATH = "results/log.md"

os.makedirs("results", exist_ok=True)

df_result = pd.DataFrame([{"ID": id, "Answer": r["answer"]} for id, r in enumerate(results)])
df_result.to_csv(OUTPUT_PATH, index=False)

# Write full responses to a markdown file
with open(LOG_PATH, "w", encoding="utf-8") as f:
    for id, r in enumerate(results):
        f.write(f"# ID: {id}\n\n")
        f.write(f"**Answer:** {r['answer']}\n\n")
        f.write(f"**Prompt:**\n\n{r['prompt']}\n\n")
        f.write(f"**Response:**\n\n{r['response']}\n\n\n")